# Statistical Validation

This phase applies statistical methods to validate patterns observed during exploratory and SQL analysis.

The objective is to confirm whether observed logistics trends are statistically significant rather than random variations.

Key tasks include:

• Reviewing probability concepts  
• Formulating null and alternative hypotheses  
• Conducting statistical tests (t-test, chi-square)  
• Interpreting p-values and statistical significance

## 1. Data Loading

The cleaned datasets generated in Phase 2 are loaded for statistical testing.

In [1]:
import pandas as pd
from pathlib import Path


from pathlib import Path

BASE = Path.cwd()

# Support running from the project root or notebooks/ folder
if not (BASE / "data").is_dir():
    if (BASE.parent / "data").is_dir():
        BASE = BASE.parent
    else:
        raise FileNotFoundError(
            "Project data folder not found. "
            "Run this notebook from the project root or notebooks folder."
        )

RAW = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
DB = BASE / "data" / "logistics.db"

# Create only the directory needed for processed datasets
PROC.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE)
print("Raw data:", RAW)
print("Processed data:", PROC)


shipment = pd.read_csv(PROC / "shipment_cleaned.csv")
inventory = pd.read_csv(PROC / "inventory_cleaned.csv")
warehouse = pd.read_csv(PROC / "warehouse_cleaned.csv")

shipment.head()

Project root: d:\Athar\LogiTech_Logistics_Analysis
Raw data: d:\Athar\LogiTech_Logistics_Analysis\data\raw
Processed data: d:\Athar\LogiTech_Logistics_Analysis\data\processed


,Shipment_ID,Customer,Warehouse_ID,Destination_Region,Dispatch_Date,Delivery_Date,Delivery_Time_Days,Weight,Carrier_Name,Shipment_Status,Delivery_Mode,Shipment_Cost,Delivery_Accuracy_Flag,Damage_Flag
0,S008616,V55555555555555_8,PLANT03,PORT09,2023-01-01,2023-01-03,2,3.066737,V444_1,Delivered,AIR,1.5252,1,0
1,S007268,V555555555555555555_17,PLANT03,PORT09,2023-01-01,2023-01-03,2,4.680013,V444_1,Delivered,AIR,1.5252,1,0
2,S007608,V5555555_22,PLANT03,PORT09,2023-01-01,2023-01-03,2,5.145045,V444_1,Delivered,AIR,1.2020,1,0
3,S008205,V5555555_19,PLANT03,PORT09,2023-01-01,2023-01-03,2,5.939471,V444_1,Delivered,AIR,9.0884,1,0
4,S008717,V55555555555555_8,PLANT03,PORT09,2023-01-01,2023-01-03,2,19.270204,V444_1,Delivered,AIR,49.4272,1,0


## 2. Probability Analysis

Basic probability measures are calculated to understand the likelihood of operational events such as shipment damage.

In [2]:
damage_probability = shipment["Damage_Flag"].mean()

print("Probability of shipment damage:", round(damage_probability,4))

Probability of shipment damage: 0.0269


• Shipment damage occurs with low probability (~2.69%), indicating generally safe logistics operations.

## 3. Chi-Square Test

H0 (Null Hypothesis): Shipment damage is NOT related to warehouse location.

H1 (Alternative Hypothesis): Shipment damage IS related to warehouse location.

In [3]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(shipment["Warehouse_ID"], shipment["Damage_Flag"])

chi2, p, dof, expected = chi2_contingency(contingency)

print("Chi-Square Statistic:", chi2)
print("P-value:", p)

#Conclusion
if p < 0.05:
    print("Conclusion: Damage depends on warehouse.")
else:
    print("Conclusion: Damage does not depend on warehouse.")

Chi-Square Statistic: 4.138435475173967
P-value: 0.6579485627193931
Conclusion: Damage does not depend on warehouse.


The chi-square test shows no statistically significant relationship between warehouse location and shipment damage (p > 0.05).

## 4. Independent T-Test

H0 (Null Hypothesis): There is NO difference in delivery time between normal and delayed shipments.

H1 (Alternative Hypothesis): Delayed shipments take longer than normal deliveries.

In [4]:
from scipy.stats import ttest_ind

on_time = shipment[shipment["Delivery_Accuracy_Flag"] == 1]["Delivery_Time_Days"].dropna()
delayed = shipment[shipment["Delivery_Accuracy_Flag"] == 0]["Delivery_Time_Days"].dropna()

t_stat, p_value = ttest_ind(on_time, delayed)

print("T-Statistic:", round(t_stat, 4))
print("P-value:", round(p_value, 4))

# Conclusion
if p_value < 0.05:
    print("Conclusion: Delayed shipments take significantly longer time than normal deliveries.")
else:
    print("Conclusion: There is no significant difference in delivery time.")

T-Statistic: -3.5096
P-value: 0.0005
Conclusion: Delayed shipments take significantly longer time than normal deliveries.


## Final Conclusion

Statistical validation was performed to support the insights identified during exploratory data analysis.

- The Chi-Square test shows no significant relationship between warehouse and shipment damage.
- The T-Test shows a significant difference in delivery time between normal and delayed shipments.

Overall, these results show that delivery delays affect delivery time, while warehouse location does not affect shipment damage.